# Project 1: Missing-Data Analysis

### ANSC 4040 | Milking-session dataset

This notebook provides a structured review of missing values in the Project 1 dataset. It moves from data loading and quality checks to missingness patterns, descriptive summaries, and visual comparisons by reproduction status.

> **Analysis goal:** Identify where information is missing, quantify the scope of the problem, and document patterns before making any decision about removing or imputing records.

**Notebook map:**

1. Load and inspect the dataset
2. Measure missing values by column and row
3. Describe missingness patterns
4. Summarize observed values
5. Compare outcomes by reproduction status
6. Record findings and next steps

## 1. Setup and data loading

In [8]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

DATA_FILENAME = "Data_set_prep_assignment_1.csv"
DATA_CANDIDATES = [
    Path(DATA_FILENAME),
    Path("files") / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    searched_paths = ", ".join(str(path) for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME}. Searched: {searched_paths}"
    )

# The first row contains the column names; pandas infers the remaining types.
df = pd.read_csv(DATA_PATH, header=0, low_memory=False)

print(f"Loaded: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print("Preview of the loaded dataset:")
display(df.head())

Loaded: Data_set_prep_assignment_1.csv
Dataset shape: 8,495,421 rows x 11 columns
Preview of the loaded dataset:


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8839528597343470980,1.00,365.00,Pregnant,2019-12-09,2.90,0.90,4.98,11.16,226,1
1,-5365332022005618386,1.00,66.00,Bred,2021-07-12,3.31,1.40,5.35,15.06,268,2
2,<NA>,NaN,NaN,NaN,2020-08-24,3.22,3.50,7.55,14.56,270,1
3,7750423760892252046,1.00,244.00,Pregnant,2019-12-04,3.90,4.10,8.40,15.33,231,3
4,<NA>,NaN,NaN,NaN,2021-03-21,4.22,5.10,9.20,13.97,198,2


## 2. Data structure and quality checks

This section establishes the dataset's shape, column names, data types, and completeness. The tables below are the primary quality-control outputs for the analysis.

In [7]:
print("Column names:")
print(df.columns.tolist())

schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
}).sort_values("missing", ascending=False)
schema

Column names:
['AnimalId', 'LactationNumber', 'DaysInMilk', 'ReproductionStatus', 'EventDate', 'Avgmilkflow', 'Flow30_60Session', 'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec', 'milking']


Column names:
['AnimalId', 'LactationNumber', 'DaysInMilk', 'ReproductionStatus', 'EventDate', 'Avgmilkflow', 'Flow30_60Session', 'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec', 'milking']


,dtype,non_null,missing,missing_pct,unique_values
DaysInMilk,float64,6794414,1701007,20.02,870
AnimalId,float64,6794418,1701003,20.02,9087
LactationNumber,float64,6794418,1701003,20.02,12
ReproductionStatus,str,6794418,1701003,20.02,4
Avgmilkflow,float64,8495249,172,0.00,219
EventDate,str,8495421,0,0.00,830
Flow30_60Session,float64,8495421,0,0.00,102
YieldFirst2Min_Session,float64,8495421,0,0.00,1675
YieldSession,float64,8495421,0,0.00,1010
DurationSession_sec,int64,8495421,0,0.00,604


In [10]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_values": missing_by_column,
    "missing_percent": missing_percent.round(2),
})

rows_with_missing = df.isna().any(axis=1)
missing_rows = df.loc[rows_with_missing].copy()
complete_rows = df.loc[~rows_with_missing].copy()

print(f"Rows with at least one missing value: {len(missing_rows):,}")
print(f"Complete rows: {len(complete_rows):,}")
print(f"Missing cells in the entire dataset: {int(df.isna().sum().sum()):,}")
missing_summary

Rows with at least one missing value: 1,701,138
Complete rows: 6,794,283
Missing cells in the entire dataset: 6,804,188


Rows with at least one missing value: 1,701,138
Complete rows: 6,794,283
Missing cells in the entire dataset: 6,804,188


,missing_values,missing_percent
DaysInMilk,1701007,20.02
AnimalId,1701003,20.02
LactationNumber,1701003,20.02
ReproductionStatus,1701003,20.02
Avgmilkflow,172,0.00
EventDate,0,0.00
Flow30_60Session,0,0.00
YieldFirst2Min_Session,0,0.00
YieldSession,0,0.00
DurationSession_sec,0,0.00


## 3. Missingness patterns

The next outputs identify incomplete rows and show whether missing fields are concentrated in particular records. Focus first on the counts and percentages, then use the sample rows to investigate possible causes.

In [11]:
missing_count_per_row = df.isna().sum(axis=1)
row_pattern_summary = (
    missing_count_per_row.value_counts()
    .sort_index()
    .rename_axis("number_of_missing_fields")
    .rename("row_count")
    .to_frame()
)

print("Rows grouped by how many fields are missing:")
display(row_pattern_summary)

print("Sample rows containing missing values:")
display(missing_rows.head(20))

print("Rows with missing identifiers or timing/status fields:")
key_columns = [
    column for column in [
        "AnimalNumber", "LactationNumber", "DaysInMilk", "ReproductionStatus"
    ] if column in df.columns
]
display(missing_rows.loc[:, key_columns + ["Avgmilkflow"]].head(20))

Rows grouped by how many fields are missing:


,row_count
number_of_missing_fields,
0,6794283
1,135
4,1700962
5,41


Sample rows containing missing values:


Rows grouped by how many fields are missing:


,row_count
number_of_missing_fields,
0,6794283
1,135
4,1700962
5,41


...

,row_count
number_of_missing_fields,
0,6794283
1,135
4,1700962
5,41


Sample rows containing missing values:


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
2,<NA>,NaN,NaN,NaN,2020-08-24,3.22,3.50,7.55,14.56,270,1
4,<NA>,NaN,NaN,NaN,2021-03-21,4.22,5.10,9.20,13.97,198,2
5,<NA>,NaN,NaN,NaN,2020-04-16,2.31,2.50,4.65,9.66,245,2
10,<NA>,NaN,NaN,NaN,2021-04-01,2.59,2.70,5.20,8.57,193,2
12,<NA>,NaN,NaN,NaN,2019-11-22,3.22,4.19,7.37,9.07,165,1
24,<NA>,NaN,NaN,NaN,2020-11-19,2.22,2.50,4.75,9.80,257,1
26,<NA>,NaN,NaN,NaN,2020-05-10,3.31,4.60,7.40,8.30,147,2
28,<NA>,NaN,NaN,NaN,2021-09-23,3.49,4.80,8.85,12.43,211,1
29,<NA>,NaN,NaN,NaN,2020-09-27,3.31,4.19,7.29,7.94,140,2
31,<NA>,NaN,NaN,NaN,2021-07-27,3.22,3.70,6.90,17.33,323,3


Rows with missing identifiers or timing/status fields:


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,Avgmilkflow
2,<NA>,NaN,NaN,NaN,3.22
4,<NA>,NaN,NaN,NaN,4.22
5,<NA>,NaN,NaN,NaN,2.31
10,<NA>,NaN,NaN,NaN,2.59
12,<NA>,NaN,NaN,NaN,3.22
24,<NA>,NaN,NaN,NaN,2.22
26,<NA>,NaN,NaN,NaN,3.31
28,<NA>,NaN,NaN,NaN,3.49
29,<NA>,NaN,NaN,NaN,3.31
31,<NA>,NaN,NaN,NaN,3.22


## 4. Context for observed values

The following summaries describe non-missing values only. They provide context after the missing cells have been identified; they do not replace or impute the missing data.

In [ ]:
numeric_columns = [
    "LactationNumber", "DaysInMilk", "Avgmilkflow", "Flow30_60Session",
    "YieldFirst2Min_Session", "YieldSession",
    "DurationSession_sec", "milking"
]
numeric_columns = [column for column in numeric_columns if column in df.columns]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

summary = df[numeric_columns].describe().T
summary["missing"] = df[numeric_columns].isna().sum()
summary.round(2)

,count,mean,std,min,25%,50%,75%,max,missing
LactationNumber,6794418.0,2.29,1.39,1.00,1.00,2.00,3.00,12.00,1701003
DaysInMilk,6794414.0,162.93,105.52,1.00,76.00,154.00,237.00,870.00,1701007
Avgmilkflow,8495249.0,3.53,0.73,0.00,2.99,3.49,3.99,23.50,172
Flow30_60Session,8495421.0,3.75,1.49,0.00,2.80,3.90,4.90,10.50,0
YieldFirst2Min_Session,8495421.0,7.49,2.01,2.00,6.02,7.53,9.00,12.00,0
YieldSession,8495421.0,13.93,3.65,5.03,11.43,13.74,16.33,54.84,0
DurationSession_sec,8495421.0,236.86,54.84,100.00,197.00,231.00,272.00,785.00,0
milking,8495421.0,2.00,0.82,1.00,1.00,2.00,3.00,3.00,0


## 5. Observed outcomes by reproduction status

These summaries compare the observed, non-missing outcomes across reproduction-status groups. Use the table for exact counts and statistics, and the boxplots for distributional differences and potential outliers.

In [ ]:
outcome_columns = [
    column for column in [
        "Avgmilkflow", "Flow30_60Session", "YieldFirst2Min_Session",
        "YieldSession", "DurationSession_sec"
    ] if column in df.columns
]
group_column = "ReproductionStatus"
grouped_summary = (
    df.groupby(group_column, dropna=False)[outcome_columns]
      .agg(["count", "mean", "median"])
)
grouped_summary.round(2)

Avgmilkflow              Flow30_60Session               \
                         count  mean median            count  mean median   
ReproductionStatus                                                          
Bred                   1844934  3.66   3.72          1844972  3.87    4.0   
Fresh                   969736  3.45   3.40           969753  3.73    3.8   
Open                    622415  3.51   3.49           622426  3.76    3.9   
Pregnant               3357202  3.48   3.49          3357267  3.69    3.9   
NaN                    1700962  3.53   3.49          1701003  3.75    3.9   

                   YieldFirst2Min_Session              YieldSession         \
                                    count  mean median        count   mean   
ReproductionStatus                                                           
Bred                              1844972  7.74   7.77      1844972  15.34   
Fresh                              969753  7.31   7.29       969753  14.35   
Open                               622426  7.39   7.38       622426  14.68   
Pregnant                          3357267  7.43   7.45      3357267  12.89   
NaN                               1701003  7.49   7.53      1701003  13.93   

                          DurationSession_sec                 
                   median               count    mean median  
ReproductionStatus                                            
Bred                15.20             1844972  252.01  248.0  
Fresh               14.20              969753  249.76  245.0  
Open                14.51              622426  249.97  247.0  
Pregnant            12.79             3357267  222.40  217.0  
NaN                 13.74             1701003  236.84  231.0

In [ ]:
plot_columns = [column for column in ["YieldSession", "Avgmilkflow", "DurationSession_sec"] if column in df.columns]
fig, axes = plt.subplots(1, len(plot_columns), figsize=(6 * len(plot_columns), 5))
axes = np.atleast_1d(axes)
for axis, column in zip(axes, plot_columns):
    sns.boxplot(data=df, x=group_column, y=column, ax=axis, color="#e09f3e")
    axis.set_title(f"{column} by reproduction status")
    axis.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 6. Findings and next steps

For this exercise, report the missing-data results first: which variables contain holes, how many values are missing, what percentage of each column is affected, how many rows are incomplete, and which fields tend to be missing together. The displayed sample rows identify concrete records that need attention.

Do not fill or delete missing values until the reason for the missingness is understood. Any later imputation or row removal should be documented separately, justified by the missingness mechanism, and rechecked against the original counts.